# Pose search with scikit-learn

The notebook version of the `/pose` route (`pose_sklearn.py`). It finds the dataset poses closest to a given 14-joint pose, using scikit-learn's `NearestNeighbors`.

**To run:** put `joints_normalized.csv` next to this notebook, install `pip install scikit-learn pandas numpy`, and run the cells from top to bottom. The settings cell is the one to change.

The joints must be in this order (the same as `pose-export.jsx`): right ankle, right knee, right hip, left hip, left knee, left ankle, right wrist, right elbow, right shoulder, left shoulder, left elbow, left wrist, neck, head top.

## 1. Settings

In [1]:
CSV_PATH = "joints_normalized.csv"   # normalized joints: one row per image per joint
MODE = "hybrid"    # "coords" (joint positions), "directions" (bone angles) or "hybrid" (both)
ALPHA = 0.5        # hybrid only: weight of the positions (1 - ALPHA goes to the bone angles)
MIRROR = False     # True also searches left-right flipped copies
TOP = 6            # how many matches to return

## 2. Constants

In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import NearestNeighbors

# Must match the order the frontend sends and the dataset
JOINT_NAMES = [
    "right_ankle", "right_knee", "right_hip", "left_hip", "left_knee", "left_ankle",
    "right_wrist", "right_elbow", "right_shoulder", "left_shoulder", "left_elbow", "left_wrist",
    "neck", "head_top",
]
IDX = {name: i for i, name in enumerate(JOINT_NAMES)}
R_HIP, L_HIP, NECK = IDX["right_hip"], IDX["left_hip"], IDX["neck"]

# child -> parent, the same tree as the mannequin
TREE = {
    "head_top": "neck", "right_shoulder": "neck", "left_shoulder": "neck",
    "right_elbow": "right_shoulder", "right_wrist": "right_elbow",
    "left_elbow": "left_shoulder", "left_wrist": "left_elbow",
    "right_hip": "neck", "left_hip": "neck",
    "right_knee": "right_hip", "right_ankle": "right_knee",
    "left_knee": "left_hip", "left_ankle": "left_knee",
}
BONES = list(TREE)
MIRROR_ORDER = [5, 4, 3, 2, 1, 0, 11, 10, 9, 8, 7, 6, 12, 13]   # swaps every left joint with its right one

## 3. Features

Every pose is first normalized (hip midpoint at the origin, neck-to-hip-midpoint = 1 unit), then turned into numbers the search can compare: joint positions, bone directions, or both.

In [3]:
def normalize(X):
    """(N, 14, 2) -> hip-centered, torso length 1. Raises ValueError for a degenerate pose."""
    hip = (X[:, R_HIP] + X[:, L_HIP]) / 2
    scale = np.linalg.norm(X[:, NECK] - hip, axis=1)
    if (scale < 1e-6).any():
        raise ValueError("neck is on the hip midpoint, cannot normalize")
    return (X - hip[:, None, :]) / scale[:, None, None]


def coords(X):
    return X.reshape(len(X), -1)                                    # 28 numbers


def directions(X):
    v = np.stack([X[:, IDX[c]] - X[:, IDX[TREE[c]]] for c in BONES], axis=1)
    v = v / (np.linalg.norm(v, axis=2, keepdims=True) + 1e-9)
    return v.reshape(len(X), -1)                                    # 26 numbers


def block_scale(F):
    """Typical size of a feature block, so two blocks can be mixed on equal footing."""
    return float(np.sqrt(((F - F.mean(0)) ** 2).sum(1).mean()))

## 4. The index

In [4]:
class PoseIndex:
    def __init__(self, names, poses, mode="hybrid", alpha=0.5, mirror=False):
        self.mode, self.alpha = mode, alpha
        db = np.concatenate([poses, poses[:, MIRROR_ORDER] * np.array([-1.0, 1.0])]) if mirror else poses
        self.names, self.n = names, len(poses)
        self._sc, self._sd = block_scale(coords(db)), block_scale(directions(db))
        features = self.features(db)
        self.nn = NearestNeighbors(algorithm="brute", metric="euclidean").fit(features)
        # distance between two unrelated poses, used to turn a distance into a 0-100 display score
        sample = features[:: max(1, len(features) // 200)]
        self.typical = float(np.median(pairwise_distances(sample, features)))

    def features(self, X):
        if self.mode == "coords":
            return coords(X)
        if self.mode == "directions":
            return directions(X)
        return np.hstack([np.sqrt(self.alpha) * coords(X) / self._sc,
                          np.sqrt(1 - self.alpha) * directions(X) / self._sd])

    def search(self, query, k):
        """query: (14, 2) normalized joints. Returns (image, distance, mirrored) for the k closest."""
        k = min(k, self.nn.n_samples_fit_)
        dist, idx = self.nn.kneighbors(self.features(query[None]), n_neighbors=k)
        return [(self.names[i % self.n], float(d), bool(i >= self.n)) for i, d in zip(idx[0], dist[0])]

## 5. Load the poses from the CSV

In [5]:
df = pd.read_csv(CSV_PATH).sort_values(["image", "joint"])
names = sorted(df["image"].unique())
poses = np.zeros((len(names), 14, 2))
for k, (_, g) in enumerate(df.groupby("image", sort=True)):
    g = g.set_index("joint")
    for j in JOINT_NAMES:
        poses[k, IDX[j]] = g.loc[j, ["x", "y"]].values
print(poses.shape, "->", len(names), "poses with 14 joints each")

(2000, 14, 2) -> 2000 poses with 14 joints each


**Optional:** load the skeletons from Supabase instead (the same query the API route uses). Set `USE_SUPABASE = True`, and set `SUPABASE_URL` and `SUPABASE_KEY` in your environment first.

In [6]:
USE_SUPABASE = False

if USE_SUPABASE:
    import os
    from supabase import create_client

    sb = create_client(os.environ["SUPABASE_URL"], os.environ["SUPABASE_KEY"])
    rows, offset = [], 0
    while True:   # Supabase returns at most 1000 rows per request
        res = sb.table("image_skeletons").select("image, joints").order("image").range(offset, offset + 999).execute()
        rows += res.data
        if len(res.data) < 1000:
            break
        offset += 1000
    names = [r["image"] for r in rows]
    poses = np.array([[[r["joints"][j]["x"], r["joints"][j]["y"]] for j in JOINT_NAMES] for r in rows])
    print(len(names), "skeletons loaded from Supabase")

## 6. Build the index and define the search

In [7]:
index = PoseIndex(names, poses, MODE, ALPHA, MIRROR)
print(f"{len(names)} poses | mode={MODE} | mirror={MIRROR} | typical distance between unrelated poses: {index.typical:.3f}")

2000 poses | mode=hybrid | mirror=False | typical distance between unrelated poses: 1.184


In [8]:
def find_similar(joints, top=TOP, idx=None):
    """joints: 14 points [x, y] or [x, y, z] in LSP order, the same thing the frontend sends."""
    idx = idx or index
    points = np.array([p[:2] for p in joints], dtype=float)      # the dataset is 2D, so z is dropped
    if points.shape != (14, 2):
        raise ValueError("need exactly 14 joints")
    query = normalize(points[None])[0]
    return [
        {"image": name, "distance": round(d, 4),
         "score": round(100 * max(0.0, 1 - d / idx.typical)),    # display number only
         "mirrored": mirrored}
        for name, d, mirrored in idx.search(query, top)
    ]


def show(results):
    print(f"{'image':<12} {'distance':>9} {'score':>6}  mirrored")
    for r in results:
        print(f"{r['image']:<12} {r['distance']:>9.4f} {r['score']:>6}  {r['mirrored']}")

## 7. Try it

**A dataset pose at a different size and position** (like the mannequin's units). It should come back as itself, with distance 0.

In [9]:
i = names.index("im0123.jpg")
show(find_similar(poses[i] * 7.3 + np.array([2.0, -5.0])))

image         distance  score  mirrored
im0123.jpg      0.0000    100  False
im0826.jpg      0.2050     83  False
im1015.jpg      0.2882     76  False
im1009.jpg      0.2977     75  False
im1799.jpg      0.3057     74  False
im0164.jpg      0.3381     71  False


**The mannequin's neutral T-pose**, built from the rest joints in `viewport.jsx`.

In [10]:
rest = {  # FBX units, the order pose-export.jsx sends. "L" is the viewer's left, which is the person's right
    "footL": (-2.75, 3), "kneeL": (-2.75, 15), "hipL": (-2.75, 26.5),
    "hipR": (2.75, 26.5), "kneeR": (2.75, 15), "footR": (2.75, 3),
    "handL": (-26.5, 41.1), "elbowL": (-17, 42.6), "shoulderL": (-6.5, 44),
    "shoulderR": (6.5, 44), "elbowR": (17, 42.6), "handR": (26.5, 41.1),
    "neck": (0, 47), "headTop": (0, 56.26),
}
t_pose = [list(p) + [0.0] for p in rest.values()]
show(find_similar(t_pose, top=5))

image         distance  score  mirrored
im0793.jpg      0.1927     84  False
im1349.jpg      0.4718     60  False
im0266.jpg      0.4737     60  False
im1205.jpg      0.4945     58  False
im1572.jpg      0.5112     57  False


**Your own pose.** Copy the `joints` array from a real request (browser dev tools, Network tab, the `POST /pose` request, Payload) and paste it below.

In [11]:
my_pose = None
# my_pose = [[x, y, z], [x, y, z], ...]   # 14 points

if my_pose:
    show(find_similar(my_pose))
else:
    print("paste a pose into my_pose to try your own")

paste a pose into my_pose to try your own


## 8. Compare the three modes on the T-pose

`coords` is the closest to the original numpy search, `directions` ignores limb lengths, and `hybrid` mixes both.

In [12]:
for mode in ("coords", "directions", "hybrid"):
    idx_mode = PoseIndex(names, poses, mode, ALPHA, MIRROR)
    top = find_similar(t_pose, top=4, idx=idx_mode)
    print(f"{mode:<11}", [(r["image"], r["score"]) for r in top])

coords      [('im0793.jpg', 79), ('im1349.jpg', 62), ('im0825.jpg', 59), ('im1205.jpg', 59)]
directions  [('im0793.jpg', 87), ('im0266.jpg', 62), ('im1075.jpg', 60), ('im1361.jpg', 60)]
hybrid      [('im0793.jpg', 84), ('im1349.jpg', 60), ('im0266.jpg', 60), ('im1205.jpg', 58)]


## 9. Mirroring

With `mirror=True`, a left-right flipped copy of every pose is searched too. A flipped copy of `im0042` should match the original and be marked `mirrored`.

In [13]:
mirror_index = PoseIndex(names, poses, MODE, ALPHA, mirror=True)
flipped = poses[names.index("im0042.jpg")][MIRROR_ORDER] * np.array([-1.0, 1.0])
show(find_similar(flipped, top=3, idx=mirror_index))

image         distance  score  mirrored
im0042.jpg      0.0000    100  True
im1787.jpg      0.5520     54  True
im1326.jpg      0.6807     43  False


## 10. A pose that cannot be normalized

In [14]:
try:
    find_similar([[0.0, 0.0, 0.0]] * 14)
except ValueError as e:
    print("rejected:", e)

rejected: neck is on the hip midpoint, cannot normalize
